# Experiment 06 — Best Configuration Synthesis (BCE Standard)

**Goal:** Combine winning strategies (8 Features, 3 Layers, Data Re-uploading) with proper Binary Cross-Entropy loss.

In [ ]:
import sys
sys.path.append('..')
import time
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import roc_auc_score, roc_curve
from utils.data_utils import load_higgs

N_FEATURES = 8
N_LAYERS = 3
N_SAMPLES = 1000

X_train, X_val, X_test, y_train, y_val, y_test = load_higgs(
    path='../data/HIGGS.csv.gz', n_samples=N_SAMPLES, n_features=N_FEATURES, scale_range=(0, np.pi)
)

dev = qml.device('lightning.qubit', wires=N_FEATURES)

@qml.qnode(dev, interface='autograd')
def circuit(weights, x):
    for l in range(N_LAYERS):
        for i in range(N_FEATURES): qml.RY(x[i], wires=i)
        for q in range(N_FEATURES): qml.Rot(*weights[l, q], wires=q)
        for q in range(N_FEATURES): qml.CNOT(wires=[q, (q + 1) % N_FEATURES])
    return qml.expval(qml.PauliZ(0))

def vqc_prob(w, x, b):
    return (circuit(w, x) + b + 1.0) / 2.0

def bce_loss(weights, bias, X, y):
    probs = pnp.array([vqc_prob(weights, x, bias) for x in X])
    probs = pnp.clip(probs, 1e-15, 1.0 - 1e-15)
    return -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))

def train():
    pnp.random.seed(42)
    w = pnp.array(pnp.random.uniform(0, 2*np.pi, (N_LAYERS, N_FEATURES, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=0.05)
    
    for epoch in range(40):
        idx = np.random.permutation(len(X_train))
        for start in range(0, len(X_train), 32):
            Xb, yb = X_train[idx][start:start+32], y_train[idx][start:start+32].astype(float)
            w, b = opt.step(lambda w_, b_: bce_loss(w_, b_, Xb, yb), w, b)
        print(f"Epoch {epoch+1} complete")
    return w, b

print("Training Synthesis 2.1...")
w_f, b_f = train()

test_probs = np.array([float(vqc_prob(w_f, x, b_f)) for x in X_test])
auc = roc_auc_score(y_test, test_probs)
print(f"\nFINAL TEST AUC: {auc:.4f}")